In [12]:
from model.model import get_model
import os
from dotenv import load_dotenv
from outlines import Generator, from_transformers  
from schema.ticket import Ticket , json_ticket
from prompt.summarizer import summary_prompt
from schema.table import Table
import json

load_dotenv()

True

In [2]:
location = os.getenv('DATA_FILE_NAME')
table_1 = Table(location)

In [3]:
name = os.getenv('MODEL_NAME')
hf_model,hf_tokenizer = get_model(name)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model downloaded


In [4]:
hf_model.eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [5]:
model = from_transformers(hf_model,hf_tokenizer)

In [6]:
generator = Generator(model,Ticket)

In [7]:
print(summary_prompt(table_1.return_ticket(1)))

Role: You are a precise ticket classification assistant.

   task: Classify the customer ticket.

   Constraints: - Return only the requested result.
- Do not provide explanations.
- Do not use markdown.
- summary must be one sentence as max

    input : I forgot my password and need to reset it.


In [20]:
result = generator(summary_prompt(table_1.return_ticket(1)), max_new_tokens=150,)
#print(result)

In [23]:
output = Ticket.model_validate_json(result)
print(output.model_dump_json())

{"category":"account","sentiment":"negative","urgency":"high","summary":"The customer needs assistance with their account login due to forgetting their password."}


In [ ]:
type(output)

schema.ticket.Ticket